In [7]:
import pandas as pd
df = pd.read_csv("Hydrogen_Uncertainty_dataset.csv")

In [8]:
df

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%),Performance Parameter: Annual Operating Hours (hours),Performance Parameter: Annual H2 Production (million tonne/yr),Performance Parameter: Annual CO2 Captured (million tonne/yr),Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2),Variable O&M Cost: CO2 T&S: O&M Cost ($/lb H2),Drift Loss (tons/hr),Evaporation (tons/hr)
0,7.975000,96.3675,76.5375,6709.276855,0.125799,1.284130,1.184908,0.065090,0.273211,437.137512
1,10.424999,95.7825,73.5625,6448.488770,0.120909,1.231241,1.412497,0.064933,0.272239,435.583099
2,7.825000,97.4025,92.6375,8120.603027,0.152261,1.560883,1.118635,0.065368,0.276588,442.541260
3,3.325000,94.2075,61.4875,5389.994141,0.101062,1.022440,0.850518,0.064511,0.270870,433.391388
4,8.525001,90.3375,84.0625,7368.918945,0.138167,1.375331,1.198089,0.063472,0.262637,420.219086
...,...,...,...,...,...,...,...,...,...,...
195,2.425000,96.4575,84.5875,7414.940430,0.139030,1.419718,0.666642,0.065114,0.273406,437.448853
196,7.675000,94.7475,87.5625,7675.728516,0.143920,1.459296,1.117000,0.064655,0.271258,434.012909
197,4.275000,98.7075,87.0375,7629.707031,0.143057,1.474381,0.828909,0.065718,0.288059,460.894867
198,2.675000,94.4775,66.9125,5865.549316,0.109979,1.113899,0.761790,0.064583,0.271064,433.702454


In [9]:
df.isna().sum()

Natural Gas Cost ($/MMBtu, HHV)                                   0
Absorber CO2 Removal Efficiency (%)                               0
Capacity Factor (%)                                               0
Performance Parameter: Annual Operating Hours (hours)             0
Performance Parameter: Annual H2 Production (million tonne/yr)    0
Performance Parameter: Annual CO2 Captured (million tonne/yr)     0
Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)    0
Variable O&M Cost: CO2 T&S: O&M Cost ($/lb H2)                    0
Drift Loss (tons/hr)                                              0
Evaporation (tons/hr)                                             0
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
# water consumption = cooling-tower evaporation + drift
df['Water Consumption (tons/hr)'] = df['Evaporation (tons/hr)'] + df['Drift Loss (tons/hr)']

# high-ambient rows zero out evaporation (infeasible tower, not real). dropping
df = df[df['Evaporation (tons/hr)'] > 0].reset_index(drop=True)

In [12]:
df

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%),Performance Parameter: Annual Operating Hours (hours),Performance Parameter: Annual H2 Production (million tonne/yr),Performance Parameter: Annual CO2 Captured (million tonne/yr),Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2),Variable O&M Cost: CO2 T&S: O&M Cost ($/lb H2),Drift Loss (tons/hr),Evaporation (tons/hr),Water Consumption (tons/hr)
0,7.975000,96.3675,76.5375,6709.276855,0.125799,1.284130,1.184908,0.065090,0.273211,437.137512,437.410723
1,10.424999,95.7825,73.5625,6448.488770,0.120909,1.231241,1.412497,0.064933,0.272239,435.583099,435.855339
2,7.825000,97.4025,92.6375,8120.603027,0.152261,1.560883,1.118635,0.065368,0.276588,442.541260,442.817848
3,3.325000,94.2075,61.4875,5389.994141,0.101062,1.022440,0.850518,0.064511,0.270870,433.391388,433.662258
4,8.525001,90.3375,84.0625,7368.918945,0.138167,1.375331,1.198089,0.063472,0.262637,420.219086,420.481723
...,...,...,...,...,...,...,...,...,...,...,...
195,2.425000,96.4575,84.5875,7414.940430,0.139030,1.419718,0.666642,0.065114,0.273406,437.448853,437.722258
196,7.675000,94.7475,87.5625,7675.728516,0.143920,1.459296,1.117000,0.064655,0.271258,434.012909,434.284167
197,4.275000,98.7075,87.0375,7629.707031,0.143057,1.474381,0.828909,0.065718,0.288059,460.894867,461.182926
198,2.675000,94.4775,66.9125,5865.549316,0.109979,1.113899,0.761790,0.064583,0.271064,433.702454,433.973518


In [13]:
x = df[['Natural Gas Cost ($/MMBtu, HHV)',
    'Absorber CO2 Removal Efficiency (%)',
    'Capacity Factor (%)']]

y = df['Water Consumption (tons/hr)'] 

In [14]:
x

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%)
0,7.975000,96.3675,76.5375
1,10.424999,95.7825,73.5625
2,7.825000,97.4025,92.6375
3,3.325000,94.2075,61.4875
4,8.525001,90.3375,84.0625
...,...,...,...
195,2.425000,96.4575,84.5875
196,7.675000,94.7475,87.5625
197,4.275000,98.7075,87.0375
198,2.675000,94.4775,66.9125


In [15]:
y

0      437.410723
1      435.855339
2      442.817848
3      433.662258
4      420.481723
          ...    
195    437.722258
196    434.284167
197    461.182926
198    433.973518
199    424.642494
Name: Water Consumption (tons/hr), Length: 200, dtype: float64

In [16]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== REGRESSION MODELS ===\n")


=== REGRESSION MODELS ===



In [17]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.8617124275969325
Test R2: 0.7984726031505697


In [18]:
# SVR
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])
svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["linear", "rbf"],
        "svr__C": [0.01, 0.1, 1, 10,100],
        "svr__gamma": ["scale"]
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(x_train, y_train)

best_svr = svr_grid.best_estimator_
print("SVR best params:", svr_grid.best_params_)
print('Train R2:',best_svr.score(x_train,y_train))
print('Test R2:',best_svr.score(x_test,y_test))

SVR best params: {'svr__C': 100, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
Train R2: 0.9916362305627514
Test R2: 0.9559093455389109


In [19]:
# XGBoost Regressor
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(x_train, y_train)

best_xgb = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)
print('Train R2:',best_xgb.score(x_train,y_train))
print('Test R2:',best_xgb.score(x_test,y_test))

XGBoost best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.9999999932001199
Test R2: 0.9969686166685184


In [20]:
# Gradient Boosting Regressor
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "max_features": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(x_train, y_train)

best_gbr = gbr_grid.best_estimator_
print("Gradient Boosting best params:", gbr_grid.best_params_)
print('Train R2:', best_gbr.score(x_train, y_train))
print('Test R2:', best_gbr.score(x_test, y_test))

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 1.0, 'n_estimators': 100, 'subsample': 1.0}
Train R2: 0.9999806325584143
Test R2: 0.9970210882074764


In [21]:
# Random Forest Regressor
rfr_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100,300],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ['sqrt',0.5, 0.7]
    },
    cv=5, scoring="r2", n_jobs=-1
)
rfr_grid.fit(x_train, y_train)

best_rfr = rfr_grid.best_estimator_
print("RF best params:", rfr_grid.best_params_)
print('Train R2:',best_rfr.score(x_train,y_train))
print('Test R2:',best_rfr.score(x_test,y_test))

RF best params: {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Train R2: 0.9966838960849235
Test R2: 0.9596671865701938


In [22]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 2, 'reg__alpha': 0.001}
Train R2: 0.9088482344008859
Test R2: 0.852531294742846


In [23]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Linear Regression | RMSE: 4.7838 | MAPE: 0.7427%
SVR | RMSE: 2.2376 | MAPE: 0.1996%
Random Forest | RMSE: 2.1401 | MAPE: 0.1887%
XGBoost | RMSE: 0.5867 | MAPE: 0.0804%
Gradient Boost | RMSE: 0.5816 | MAPE: 0.0732%
Polynomial | RMSE: 4.0922 | MAPE: 0.7810%


In [24]:
#Exporting to excel
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file       = 'Hydrogen_Water_Consumption.xlsx'
experiment = 'Target_4_Water_Consumption'   # change per experiment

results = []

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Experiment': experiment,
        'Model':      name,
        'Train R2':   round(model.score(x_train, y_train), 4),
        'Test R2':    round(model.score(x_test, y_test), 4),
        'MAE':        round(mean_absolute_error(y_test, y_pred), 4),
        'MSE':        round(mean_squared_error(y_test, y_pred), 4),
        'RMSE':       round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE (%)':   round(np.mean(np.abs((y_test - y_pred) / y_test)) * 100, 4)
    })

results_df = pd.DataFrame(results)

if os.path.exists(file):
    existing = pd.read_excel(file)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_excel(file, index=False)
else:
    results_df.to_excel(file, index=False)

print(results_df)

                   Experiment              Model  Train R2  Test R2     MAE  \
0  Target_4_Water_Consumption  Linear Regression    0.8617   0.7985  3.3045   
1  Target_4_Water_Consumption                SVR    0.9916   0.9559  0.9075   
2  Target_4_Water_Consumption      Random Forest    0.9967   0.9597  0.8528   
3  Target_4_Water_Consumption            XGBoost    1.0000   0.9970  0.3588   
4  Target_4_Water_Consumption     Gradient Boost    1.0000   0.9970  0.3277   
5  Target_4_Water_Consumption         Polynomial    0.9088   0.8525  3.4366   

       MSE    RMSE  MAPE (%)  
0  22.8845  4.7838    0.7427  
1   5.0067  2.2376    0.1996  
2   4.5800  2.1401    0.1887  
3   0.3442  0.5867    0.0804  
4   0.3383  0.5816    0.0732  
5  16.7459  4.0922    0.7810  
